In [ ]:
import pandas as pd

# Load the base dataset
df_balance = pd.read_csv('BALANCE.csv')

# --- STRATIFIED SAMPLING LOGIC ---
# This ensures rare long-stay cases (up to 180 days) are preserved 
# while downsampling common short-stay cases.
pieces = []

# Phase 1: Short Stay Baseline (0-3 Days) - 20% Sample
df_0_3 = df_balance[df_balance['los'].isin([0, 1, 2, 3])]
pieces.append(df_0_3.sample(frac=0.20, random_state=42))

# Phase 2: The Transition Ramp (4-8 Days)
# Gradually increasing sampling fraction to preserve more variance
for los_val, fraction in zip(range(4, 9), [0.25, 0.30, 0.40, 0.60, 0.80]):
    pieces.append(df_balance[df_balance['los'] == los_val].sample(frac=fraction, random_state=42))

# Phase 3: Long Stay Full Census (9-180 Days) - 100% Preservation
# This preserves the "tail" described in Figure 3
pieces.append(df_balance[df_balance['los'] >= 9])

# --- COMBINE & EXPORT ---
df_for_filter = pd.concat(pieces)
df_for_filter.to_csv('FOR_FILTER.csv', index=False)

print(f"Sampled Dataset Size: {len(df_for_filter):,}")